In [25]:
from platform import python_version
print(python_version())

3.11.14


### Calculating all possible Enrichment Analysis
  - for each LFC/FDR cutoff one calculates the Enrichment Analysis
  - We used Enricher python API
     - Reactome (2024)  <<--------------- only Reactome
     - Bioplanet (2019)
     - WikiPathways (2021 Human)
     - KEGG (2021 Human)
     - GO Biological Process (2021)
     - MSigDB Hallmark (2020)
   
### For each enriched pathways one calculates:
  - DEGs in the pathway
  - DEGs not in the pathway
  - TOI1, 2, 3, 4

### An index measures the trade-off between "LFC" and "Enriched Pathways" cufoff -> LFC - Enriched Pathway Trade-Off Method (LEPTOM)

  - We proposed and calculated the following possible indexes:

<p style="font-size: 20px; color: yellow;">
$index1 = \sqrt{-log{_{10}}{FDR_{pathway}} * \frac{n}{N} }$ </p>

<p style="font-size: 20px; color: cyan;">
$index2 = \sqrt{-log{_{10}}{FDR_{LFC}} * -log{_{10}}{FDR_{pathway}} }$ </p>

<p style="font-size: 20px; color: orange;">
$index3 = (-log{_{10}}{FDR_{LFC}} * -log{_{10}}{FDR_{pathway}} * \frac{n}{N})^{1/3}$ </p>

<p style="font-size: 20px; color: pink;">
$index4 = (abs\_LFC * -log{_{10}}{FDR_{LFC}} * -log{_{10}}{FDR_{pathway}} * \frac{n}{N})^{1/4}$ </p>

where,
  - n is the number of DEGs/DAPs found in the pathway
  - N is the total number of annotated DEGs/DAPs in the pathway (depend in the database, our default database is Reactome 2022)


#### Where
  - $FDR_{pathway}$ is the FDR from the enriched pathway
  - n - number of DEGs in the pathway
  - N - number of Genes annotated in the pathway by respective Dabase (Like Reactome, KEGG, etc)

In [26]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.dashcyto_lib import DASH_CYTO
from libs.calc_degs_lib import CALC_DEGS
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [ ]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

root_project = create_dir(root0_data, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={ptw_min_num_of_degs_cut}")

In [ ]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root0_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(enr.echo_parameters())


In [ ]:
enr.root_kegg, enr.kegg_fname

In [ ]:
enr.enr_db_list

In [ ]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

### Cutoffs and Results

In [ ]:
for case in case_list:
    ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)

    print(f"For {case}")
    print(f"\tLFC cutoffs: lfc={enr.LFC_cut:.3f}; fdr={enr.lfc_FDR_cut} #{enr.s_deg_dap}s = {len(degs)}")
    print(f"\tPathway cutoffs: fdr={enr.ptw_FDR_cut:.3f}; num of genes={enr.ptw_min_num_of_degs_cut}, #Pathways = {enr.n_pathways}, #{enr.s_deg_dap}s in pathwyas = {enr.n_degs_in_pathways}\n")
    

In [ ]:
fname, fname_ori, title = enr.set_lfc_names()
fname, title, enr.root_enrich

In [ ]:
enr.set_enrichment_name()

### Testing EnrichR API 

In [ ]:
case = case_list[0]
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, verbose=False)
ret, len(degs), enr.n_degs

In [ ]:
enr.set_db(geneset_num=0)

In [ ]:
len(degs)

In [ ]:
np.unique(enr.dflfc.biotype)

In [ ]:
MAX_GENES_ENRICHR_SAFE = 2000

biotype_annot = ['protein_coding','IG_C_gene', 'IG_D_gene', 'IG_J_gene',
       'IG_V_gene', 'TR_C_gene', 'TR_D_gene', 'TR_J_gene', 'TR_V_gene',
       'lncRNA', 'miRNA', 'misc_RNA', 'rRNA', 'tRNA', 'scaRNA', 'snRNA', 'snoRNA',]


dflfc = enr.dflfc

df2 = dflfc[dflfc.biotype.isin(biotype_annot)]
df2 = df2.sort_values('fdr', ascending=True)
print(len(df2))
df2.head(6)

In [ ]:
degs_to_reactome = df2.symbol.to_list()[:MAX_GENES_ENRICHR_SAFE]
len(degs_to_reactome)

In [ ]:
shortId, userListId = enr.open_session_upload_symbols(degs_to_reactome)
shortId, userListId

### Calc all enrichment analyses

In [ ]:
enr.set_db(0, verbose=True)
enr.set_enrichment_name()

### Reactome enrichment analysis

In [ ]:
enr.enr_db_list

In [ ]:
MAX_GENES_ENRICHR_SAFE = 2000

biotype_annot = ['protein_coding','IG_C_gene', 'IG_D_gene', 'IG_J_gene',
       'IG_V_gene', 'TR_C_gene', 'TR_D_gene', 'TR_J_gene', 'TR_V_gene',
       'lncRNA', 'miRNA', 'misc_RNA', 'rRNA', 'tRNA', 'scaRNA', 'snRNA', 'snoRNA',]


dflfc = enr.dflfc

df2 = dflfc[dflfc.biotype.isin(biotype_annot)]
df2 = df2.sort_values('fdr', ascending=True)
print(len(df2))
df2 = df2.head(MAX_GENES_ENRICHR_SAFE)
print(len(df2))

lista = df2.symbol.to_list()
lista = np.unique(lista)
print(len(lista))

write_txt("\n".join(lista), "degs.txt")


df2.head(6)

In [ ]:
lista

In [ ]:
force=True
verbose=True

enr.calc_EA_dataset_symbol(list(lista), calc_many_sig=False, default=False, force=force, verbose=verbose)

df_enr = enr.df_enr

print(len(df_enr))
df_enr.head(10)

### Development & tests

In [ ]:
import requests
import json

lista = list(lista)

shortId, userListId = enr.open_session_upload_symbols(lista)

if shortId == '' or userListId == '':
    print(f"Problems in open_session_upload_symbols().")
    raise ValueError("erro1")
    # return pd.DataFrame()

# print(f"Enrichr web service: for '{filefull}'")
try:
    response = requests.get( enr.ENRICHR_URL_query + enr.ENRICHR_query_string%(userListId, enr.geneset_lib) )
except:
    print(f"Problems in request geneset {enr.geneset_lib} and userListId {str(userListId)}.")
    raise ValueError("erro2")
    # return pd.DataFrame()

if not response.ok:
    print('Error fetching enrichment results: %s in %s'%(userListId, enr.geneset_lib))
    raise ValueError("erro3")
    # return pd.DataFrame()

try:
    data = json.loads(response.text)
except:
    print('Error getting data: loading json response.')
    raise ValueError("erro4")
    # return pd.DataFrame()

data[enr.geneset_lib]

In [ ]:
enr.calc_EA_dataset_symbol(list(lista), calc_many_sig=False, default=False, force=force, verbose=verbose)

df_enr = enr.df_enr

print(len(df_enr))
df_enr.head(10)

In [ ]:
mats = data[enr.geneset_lib]

dic = {}
for mat in mats:
    # print(mat)
    count = mat[0]
    desc  = mat[1]
    pval2  = mat[2]
    odds_ratio = mat[3]
    combined_score = mat[4]
    genes  = mat[5]
    fdr2  = mat[6]
    # unk0  = mat[7]
    # unk1  = mat[8]

    dic[count] ={}
    dic2 = dic[count]

    if enr.geneset_lib == 'Reactome_2022':
        # Neuronal System R-HSA-112316
        mat = desc.split(' ')
        pathway = " ".join(mat[:-1])
        _id  = mat[-1]
        dic2['pathway'] = pathway
        dic2['pathway_id'] = _id
    elif enr.geneset_lib.startswith('GO_'):
        # Positive Regulation of Cellular Process (GO:0048522)
        mat = desc.split(' ')
        pathway = " ".join(mat[:-1])
        _id  = mat[-1].strip().replace(')','').replace('(','')
        dic2['pathway'] = pathway
        dic2['pathway_id'] = _id
    elif enr.geneset_lib.startswith('WikiPathways_'):
        # Small Cell Lung Cancer WP4658
        mat = desc.split(' ')
        pathway = " ".join(mat[:-1])
        _id  = mat[-1].strip()
        dic2['pathway'] = pathway
        dic2['pathway_id'] = _id
    else:
        dic2['pathway'] = desc

    dic2['pval'] = pval2
    dic2['fdr'] = fdr2
    dic2['odds_ratio'] = odds_ratio
    dic2['combined_score'] = combined_score
    dic2['genes'] = genes
    dic2['num_of_genes'] = len(genes)

df_enr = pd.DataFrame(dic).T

In [ ]:
df_enr

In [ ]:
enr.root_reactome

In [ ]:
df_reactome = enr.open_reactome_pathway()
df_reactome.head(3)

In [ ]:
df_enr.columns

In [ ]:
fname, _ = enr.set_enrichment_name()  # fname_cutoff
fname

In [ ]:
df_reactome = enr.open_reactome_pathway()
df_reactome.head(3)

In [ ]:


df_enr = pd.merge(df_enr, df_reactome[ ['pathway', 'pathway_id'] ], how="outer", on='pathway')
cols = ['pathway', 'pathway_id', 'pval', 'fdr', 'odds_ratio', 'combined_score', 'genes', 'num_of_genes']
df_enr = df_enr[cols]
df_enr = df_enr[ ~pd.isnull(df_enr.fdr) ]

df_enr = df_enr.sort_values(['fdr', 'num_of_genes'], ascending=[True, False])
df_enr.reset_index(inplace=True, drop=True)
_ = pdwritecsv(df_enr, fname, enr.root_enrich, verbose=verbose)

In [ ]:
force=False
verbose=False
ptw_min_num_of_degs_cut = 3

geneset_num_list = [0, 1, 2, 3, 6]
geneset_num_list = [0]

icount=-1
for case in case_list:
    if not enr.open_case_simple(case):
        print(f"Problems for {case} !!!!")
        continue
    
    dfsim2 = dfsim[ (dfsim.normalization == enr.normalization) & (dfsim.case == case) &
                    (dfsim.n_degs >= ptw_min_num_of_degs_cut)]
    
    for i in range(len(dfsim2)):
        icount += 1
        
        row = dfsim2.iloc[i]

        degs = eval(row.degs)
        case = row.case
        
        LFC_cut = row.LFC_cut
        lfc_FDR_cut = row.lfc_FDR_cut

        degs2, _ = enr.list_of_degs_params(LFC_cut, lfc_FDR_cut, verbose=False)

        if len(degs) != len(degs2):
            print("Error:", case, LFC_cut, lfc_FDR_cut, len(degs), len(degs2))
            continue

        # if i > 10:break
        enr.calc_EA_dataset_symbol(degs, return_value=True, force=force, verbose=verbose)
        if icount%100==0:
            print(case, len(degs), LFC_cut, lfc_FDR_cut)
            enr.echo_degs()
            print("")
            enr.echo_enriched_pathways()
            print("\n")


### Reactome, Bioplanet, KEGG

In [ ]:
enr.enr_db_list

In [ ]:
[enr.enr_db_list[i] for i in [0, 1, 2, 3, 5, 6]]

In [ ]:
enr.set_which_db('xxx')

In [ ]:
enr.set_which_db('Reactome_2022')

In [ ]:
enr.set_which_db('Reactome')

In [ ]:
enr.set_which_db('reactome')

In [ ]:
enr.set_which_db('KEGG_2021')

In [ ]:
enr.set_which_db('KEGG')

In [ ]:
enr.set_db(geneset_num = 0)

### Start with Reactome

In [ ]:
case_list

In [ ]:
case = case_list[0]
enr.open_case(case, verbose=False)
degs, degs_ensembl, dflfc = enr.list_of_degs(force=False, verbose=True)

# enr.set_which_db('Reactome_2022')
geneset_num = 0
force=False; verbose=False

print(f"case: {case}")
print(f"there are {len(degs)} for fdr < {enr.lfc_FDR_cut:.3f} and lfc >= {enr.LFC_cut:.3f}")
print(f"Pathway filter fdr < {enr.ptw_FDR_cut:.3f} and lfc >= {enr.pathway_pval_cutoff:.3f} and ptw_min_num_of_degs_cut={enr.ptw_min_num_of_degs_cut}")

enr.calc_EA_dataset_symbol(degs, force=force, verbose=verbose)

In [ ]:
i=0
case = case_list[i]
ret, degs, dfdegs = enr.open_case(case, verbose=False)

enr.df_enr

In [ ]:
print(enr.geneset_lib)

row = enr.get_enriched_pathway_line(i_line=0)

if row is not None:
    print(row.pathway, row.pathway_id)
    print(len(enr.genes_in_pathway), enr.genes_in_pathway)


### Enriched DEGs

In [ ]:
len(enr.all_enr_degs), ",".join(enr.all_enr_degs)

In [ ]:
",".join(enr.degs)

In [ ]:
enr_found_degs = [x for x in enr.degs if x in enr.all_enr_degs]
",".join(enr_found_degs)

### Found genes 

In [ ]:
len(enr.all_enr_degs), ",".join(enr.all_enr_degs)

In [ ]:
len(enr.enr_found_degs), ",".join(enr.enr_found_degs)

### Not found genes in pathways

In [ ]:
len(enr.enr_not_found_degs), ",".join(enr.enr_not_found_degs)

### BioPlanet

In [ ]:
enr.set_db(geneset_num=3)
enr.geneset_lib

In [ ]:
fname = 'pathway.tsv'
df_biop = pdreadcsv(fname, enr.root_bioplanet)
df_biop.head(3)

In [ ]:
dfbiop = enr.open_db_pathway()
print(len(dfbiop))
dfbiop.head(3)

### Bioplanet diseases

In [ ]:
_ = enr.open_bioplanet_disease(force=False)
dfdisease = enr.dfdisease
print(len(dfdisease))
dfdisease.head()

### Bioplanet Categories

In [ ]:
 _ = enr.open_bioplanet_category(force=False)
dfbiop_cat = enr.dfbiop_cat
print(type(dfbiop_cat.category))
print(len(dfbiop_cat))
dfbiop_cat.head()

In [ ]:
def all_equal_list(cols1, cols2):
    if cols1 is None and cols2 is None: return True
    if cols1 == [] and cols2 == []: return True
    
    if len(cols1) != len(cols2): return False
    
    soma = np.sum([1 if cols1[i] != cols2[i] else 0 for i in range(len(cols1))])
    return soma == 0

cols1 = list(enr.dflfc_all.columns)

cols2 = ['probe', 'symbol', 'geneid', 'description', 'logFC', 'meanExpr', \
        't.stat', 'p-value', 'fdr', 'B', 'chr.range', 'org.chromosome', \
        'forward.reverse', 'nuc.sequence', 'gemmaid', 'go.term']

all_equal_list(cols1, cols2)

In [ ]:
all_genes = []
for i in range(len(dfsig)):
    genes = eval(dfsig.iloc[i].genes)
    # print(i, len(genes))
    all_genes += genes
    
all_genes = np.unique(all_genes)
all_genes.sort()
all_genes

not_found_genes = np.array([x for x in enr.deg_list if not x in all_genes])
not_found_genes

In [ ]:
def find_hugo_symbol(gene):
    try:
        i = enr.gene.dic_genes[gene]
        if isinstance(i, list):
            i = i[0]
            
        mat = enr.gene.df_synonyms.iloc[i]['synonyms']
        print(">>>", gene, i, mat, type(mat))
        if isinstance(mat, str):
            mat = eval(mat)
            
        gene0 = mat[0]
    except:
        gene0 = gene

    return gene0

In [ ]:
gene = 'SEPP1'
find_hugo_symbol(gene)

In [ ]:
enr.dic_genes[gene]

In [ ]:
lista = [x for x in os.listdir(enr.root_result) if 'taubate_PAC_UP_' in x and not '~lock' in x ]
print(len(lista))
", ".join(lista)


In [ ]:
root = enr.root_result
_type = '.tsv'
_type = None
pattern_src = 'taubate_PAC'
pattern_dst = 'taubate_MAP'

if _type is None:
    lista = [x for x in os.listdir(root) if pattern_src in x and not '~lock' in x ]
else:
    lista = [x for x in os.listdir(root) if pattern_src in x and not '~lock' in x and x.endswith(_type)]

if lista == []:
    print(f"There are no files in {root}")

for fname in lista:
    fname_src = os.path.join(root, fname)

    fname_dst = fname.replace(pattern_src, pattern_dst)
    fname_dst = os.path.join(root, fname_dst)

    if not  os.path.exists(fname_src):
        print(f"fname does not exist: '{fname_src}'", fname_src)
        continue
    if os.path.exists(fname_dst):
        print(f"fname already exists: '{fname_dst}'")
        continue
        
    if fname_dst == fname_src:
        print(f"fname did not change: 'fname_src'")
        continue
        
    print(fname_src, 'to', fname_dst)
    os.rename(fname_src, fname_dst)

In [ ]:

def open_enriched_table_manually(self, fname, root):
    enr.df_enr0, enr.df_enr = None, None
    fname_enr, fname_enr_cutoff = enr.set_enrichment_name()

    enr.fname_enr = fname_enr
    enr.fname_enr_cutoff = fname_enr_cutoff
    enr.root_enr = root

    filefull = os.path.join(root, fname_enr)

    if not os.path.exists(filefull):
        enr.deg_list = []
        print("File does not exists '%s'"%(filefull))
        return None

    df_enr0 = pdreadcsv(fname_enr, enr.root_enr)
    ret = False if df_enr0 is None or df_enr0.empty else True

    if ret:
        df_enr0.columns = enr.old_pathway_cols
        df_enr0 = df_enr0[enr.sel_pathway_cols]

        df_enr = df_enr0[ (df_enr0.fdr_pathway_cutoff < 0.05) & (df_enr0.num_of_genes >= enr.ptw_min_num_of_degs_cut)]

        enr.df_enr0 = df_enr0
        enr.df_enr = df_enr

        print(f">>> df_enr has {len(df_enr)} enrichment pathways and max(fdr_pathway_cutoff) = {df_enr.fdr_pathway_cutoff.max():%.3f}")
        all_enr_genes = []
        for i in range(len(df_enr)):
            s_genes = df_enr.iloc[i].genes
            if isinstance(s_genes, str) and s_genes != '':
                genes = list(s_genes.split(';'))
                all_enr_genes += genes

        all_enr_genes = list(np.unique(all_enr_genes))
    else:
        enr.df_enr0 = None
        enr.df_enr = None
        all_enr_genes = []

    enr.all_enr_genes = all_enr_genes
    return ret

### Studied symbos in pubmed

In [ ]:
prefix = "taubate_covid19"
inidate="2019/10/01"
enddate="2030/12/31"

email = "flalix@gmail.com"

sleep_entrez = [30, 90, 300]; retmax=100000,

nlp = NLP(email, prefix, root0, 
          sleep_entrez = [30, 90, 300], retmax=100000,
          only_title_abstract=True, text_quote='',
          inidate=inidate, enddate=enddate, root_colab=root_colab, 
          force_query = False, verbose_query=False, dec_ncpus=2)

df_my_gene, df_my_gene_syn = nlp.gene.open_my_gene()
print(df_my_gene.shape)
df_my_gene.head(3)

In [ ]:
dfsymb_perc = nlp.find_symbols_build_perc_table(force=False)
print(len(dfsymb_perc))
dfsymb_perc.head(3)

In [ ]:
dfsymb_perc[dfsymb_perc.symbol == 'VWF']

In [ ]:
all_pubmed = list(dfsymb_perc.symbol)
print(len(all_pubmed))
all_pubmed.sort()

degs_notin = [x for x in degs if not x in all_pubmed]
degs_notin.sort()

print(len(degs_notin))
", ".join(degs_notin)

In [ ]:
dfsymb_perc[dfsymb_perc.symbol == 'CLEC3B']

In [ ]:
not 'VWF' in all_pubmed

In [ ]:
all_pubmed = list(dfsymb_perc.symbol)
print(len(all_pubmed))
all_pubmed.sort()

# ", ".join(all_pubmed)

In [ ]:
geneset_num_list = [0, 1, 2, 4, 5, 7]
prompt_verbose=False

dic={}; i=-1
for geneset_num in geneset_num_list:
    enr.set_db(geneset_num, verbose=True)

    s_start = f"enricheR_{enr.geneset_lib}"

    for case in case_list:
        files = [x for x in os.listdir(enr.root_enrichment) if x.startswith(s_start) and case in x]
        if prompt_verbose: print("\tcase", case, len(files))

        i+=1
        dic[i] = {}
        dic2 = dic[i]
    
        dic2['geneset_num'] = enr.geneset_num
        dic2['geneset_lib'] = enr.geneset_lib
        dic2['case'] = case
        dic2['n'] = len(files)

    if prompt_verbose: print('')

dfa = pd.DataFrame(dic).T
dfa.head(3)

In [ ]:
title = f'Sampling cutoffs per geneset'
yaxis_title = f'number of samples'
width=1100
height=700

colors = ['green', 'red', 'blue', 'brown', 'yellow', 'cyan', 'lightgreen', 'pink', 'gray', 'lightblue']

geneset_lista = dfa.geneset_lib.unique()
colors_geneset = colors[:len(geneset_lista)]

_n_rows = int(np.ceil(len(enr.case_list)/2))
fig = make_subplots(rows=_n_rows, cols=2, subplot_titles=enr.case_list)

nrow=1
ncol=0

for case in enr.case_list:
    
    dfa2 = dfa[dfa.case == case].copy()
    dfa2 = dfa2.sort_values('geneset_lib')
    dfa2.index = np.arange(0, len(dfa2))

    ncol += 1
    if ncol == 3:
        ncol = 1
        nrow += 1
        
    
    fig.add_trace(go.Bar(x=dfa2.geneset_lib, y=dfa2.n, marker_color=colors_geneset, name=case), row=nrow, col=ncol)

fig.update_layout(
            autosize=True,
            title=title,
            width=width,
            height=height*_n_rows,
            plot_bgcolor='lightgray',
            xaxis_title="cases",
            yaxis_title=yaxis_title,
            showlegend=False,
            font=dict(
                family="Arial",
                size=14,
                color="Black"
            )
)

figname = title_replace(title)
figname = os.path.join(enr.root_figure, figname+'.html')

fig.write_html(figname)
if verbose: print(">>> HTML and png saved:", figname)
fig.write_image(figname.replace('.html', '.png'))

fig.show()

In [ ]:
dfa2